# 00. Extracción, Curación y Consolidación de Datos Meteorológicos Diarios (CABA 2018–2021)
## Cátedra de Metodología de la Investigación — Licenciatura en Ciencia de Datos (UGR)
### Grupo 36: Marco Fouad Abboud, Matías Gabriel Gallardo, Néstor Fabián León, Sebastián Luna

---
### Objetivos de este Cuaderno:
1. **Resolución del Requerimiento Metodológico:** Sustituir la métrica de lluvia mensual por series **diarias oficiales** sin datos faltantes.
2. **Extracción Multifuente:**
   - **ECMWF ERA5 (Copernicus C3S - Unión Europea):** Reanálisis global de alta resolución.
   - **NASA POWER (MERRA-2 - EE.UU.):** Asimilación satelital y de radar para las coordenadas exactas de CABA (-34.6037, -58.3816).
   - **Meteostat / SMN (Estación SABA - Aeroparque Jorge Newbery - OMM 87582):** Registro terrestre en superficie.
3. **Ingeniería de Atributos de Control:** Días hábiles, fines de semana, feriados oficiales de Argentina (Ley Nacional) y choque sanitario COVID-19 (ASPO).
4. **Exportación y Custodia de Linaje:** Guardado en CSV con cálculo de hash SHA-256 para trazabilidad científica.

## 1. Configuración del Entorno y Librerías

In [ ]:
import os
import hashlib
from datetime import date
from pathlib import Path
import requests
import pandas as pd
import holidays
from meteostat import daily

# Parámetros del estudio
START_DATE = "2018-01-01"
END_DATE = "2021-12-31"
LAT, LON = -34.6037, -58.3816  # Coordenadas Obelisco / Centro de CABA
print(f"Parámetros configurados: Cuatrienio {START_DATE} a {END_DATE} | Lat: {LAT}, Lon: {LON}")

Parámetros configurados: Cuatrienio 2018-01-01 a 2021-12-31 | Lat: -34.6037, Lon: -58.3816


## 2. Descarga de ECMWF ERA5 (Copernicus Climate Change Service)
Descargamos las series diarias de precipitación (`precipitation_sum`), lluvia (`rain_sum`) y temperaturas (media, máxima y mínima).

In [ ]:
url_era5 = (
    f"https://archive-api.open-meteo.com/v1/archive?"
    f"latitude={LAT}&longitude={LON}&"
    f"start_date={START_DATE}&end_date={END_DATE}&"
    f"daily=precipitation_sum,rain_sum,temperature_2m_mean,temperature_2m_max,temperature_2m_min&"
    f"timezone=America%2FArgentina%2FBuenos_Aires"
)
headers = {"User-Agent": "UGR-Research-Project/1.0"}
r_era5 = requests.get(url_era5, headers=headers, timeout=30)
r_era5.raise_for_status()

df_era5 = pd.DataFrame(r_era5.json()["daily"])
df_era5["fecha"] = pd.to_datetime(df_era5["time"])
df_era5 = df_era5.drop(columns=["time"]).rename(columns={
    "precipitation_sum": "prcp_mm_era5",
    "rain_sum": "lluvia_mm_era5",
    "temperature_2m_mean": "temp_media_c_era5",
    "temperature_2m_max": "temp_max_c_era5",
    "temperature_2m_min": "temp_min_c_era5"
})
print(f"ERA5 descargado exitosamente: {len(df_era5)} días, {df_era5.isnull().sum().sum()} nulos.")
df_era5.head()

ERA5 descargado exitosamente: 1461 días, 0 nulos.


,prcp_mm_era5,lluvia_mm_era5,temp_media_c_era5,temp_max_c_era5,temp_min_c_era5,fecha
0,0.0,0.0,22.1,25.8,17.7,2018-01-01
1,0.6,0.6,25.3,31.5,20.2,2018-01-02
2,0.0,0.0,25.8,32.6,18.6,2018-01-03
3,0.6,0.6,28.8,36.2,22.6,2018-01-04
4,4.4,4.4,22.8,27.8,19.7,2018-01-05


## 3. Descarga de NASA POWER (MERRA-2 Reanalysis)
Descargamos las variables satelitales corregidas: `PRECTOTCORR` (precipitación corregida en mm), `T2M`, `T2M_MAX`, `T2M_MIN` y `RH2M` (humedad relativa).

In [ ]:
url_nasa = (
    f"https://power.larc.nasa.gov/api/temporal/daily/point?"
    f"parameters=PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,RH2M&community=AG&"
    f"longitude={LON}&latitude={LAT}&"
    f"start={START_DATE.replace('-', '')}&end={END_DATE.replace('-', '')}&format=JSON"
)
r_nasa = requests.get(url_nasa, timeout=30)
r_nasa.raise_for_status()
params = r_nasa.json()["properties"]["parameter"]
df_nasa = pd.DataFrame(params).reset_index()
df_nasa["fecha"] = pd.to_datetime(df_nasa["index"], format="%Y%m%d")
df_nasa = df_nasa.drop(columns=["index"]).rename(columns={
    "PRECTOTCORR": "prcp_mm_nasa",
    "T2M": "temp_media_c_nasa",
    "T2M_MAX": "temp_max_c_nasa",
    "T2M_MIN": "temp_min_c_nasa",
    "RH2M": "humedad_rel_pct_nasa"
})
print(f"NASA POWER descargado exitosamente: {len(df_nasa)} días, {df_nasa.isnull().sum().sum()} nulos.")
df_nasa.head()

NASA POWER descargado exitosamente: 1461 días, 0 nulos.


,prcp_mm_nasa,temp_media_c_nasa,temp_max_c_nasa,temp_min_c_nasa,humedad_rel_pct_nasa,fecha
0,0.00,22.98,24.97,21.12,65.00,2018-01-01
1,0.00,25.31,29.37,21.05,56.96,2018-01-02
2,0.04,26.10,30.21,21.64,54.65,2018-01-03
3,0.00,28.33,32.71,23.48,59.98,2018-01-04
4,10.05,23.83,25.53,21.28,73.93,2018-01-05


## 4. Consulta a Meteostat (Estación SABA - Aeroparque)
Obtenemos los registros sinópticos de superficie de la estación oficial Aeroparque SABA (OMM 87582).

In [ ]:
try:
    ts_saba = daily("87582", date(2018, 1, 1), date(2021, 12, 31)).fetch()
    ts_saba = ts_saba.reset_index().rename(columns={"time": "fecha"})
    ts_saba["fecha"] = pd.to_datetime(ts_saba["fecha"])
    df_saba = ts_saba[["fecha", "temp", "tmin", "tmax", "prcp", "rhum", "wspd", "pres"]].rename(columns={
        "temp": "temp_media_c_saba",
        "tmin": "temp_min_c_saba",
        "tmax": "temp_max_c_saba",
        "prcp": "prcp_mm_saba",
        "rhum": "humedad_pct_saba",
        "wspd": "viento_kmh_saba",
        "pres": "presion_hpa_saba"
    })
    print(f"SABA descargado: {len(df_saba)} registros.")
except Exception as e:
    print(f"Advertencia al consultar Meteostat: {e}")
    df_saba = pd.DataFrame({"fecha": df_era5["fecha"]})
df_saba.head()

SABA descargado: 1460 registros.


,fecha,temp_media_c_saba,temp_min_c_saba,temp_max_c_saba,prcp_mm_saba,humedad_pct_saba,viento_kmh_saba,presion_hpa_saba
0,2018-01-01,22.1,18.2,<NA>,0.0,61,16.3,1010.5
1,2018-01-02,25.4,20.1,32.0,<NA>,47,13.3,1005.9
2,2018-01-03,26.2,19.5,33.0,<NA>,39,14.7,1005.2
3,2018-01-04,29.3,22.5,35.2,<NA>,45,17.3,1003.9
4,2018-01-05,23.3,19.6,28.0,<NA>,69,13.3,1007.9


## 5. Fusión e Ingeniería de Atributos Metodológicos
Consolidamos las fuentes y agregamos las variables de control teóricas requeridas:
- **Calendario Laboral:** Días hábiles, fines de semana y feriados nacionales de Argentina.
- **Choque Sanitario COVID-19:** Aislamiento Social Preventivo y Obligatorio (ASPO: 20/03/2020 a 08/11/2020).
- **Categorización Operativa de Lluvia:** Umbrales OMM ($>0.1\text{ mm}$, $\ge 5\text{ mm}$ y $\ge 20\text{ mm}$).

In [ ]:
df = pd.merge(df_era5, df_nasa, on="fecha", how="outer")
if not df_saba.empty and "fecha" in df_saba.columns:
    df = pd.merge(df, df_saba, on="fecha", how="left")
df = df.sort_values("fecha").reset_index(drop=True)

# Calendario oficial de Argentina
ar_holidays = holidays.Argentina(years=[2018, 2019, 2020, 2021])
df["dia_semana"] = df["fecha"].dt.dayofweek
df["es_fin_de_semana"] = df["dia_semana"].isin([5, 6]).astype(int)
df["es_feriado"] = df["fecha"].isin(ar_holidays).astype(int)
df["es_dia_habil"] = ((df["es_fin_de_semana"] == 0) & (df["es_feriado"] == 0)).astype(int)

# Choque ASPO COVID-19
df["es_covid_aspo"] = ((df["fecha"] >= "2020-03-20") & (df["fecha"] <= "2020-11-08")).astype(int)

# Variables estandarizadas primarias
df["precipitacion_mm"] = df["prcp_mm_era5"]
df["temperatura_media_c"] = df["temp_media_c_era5"]
df["temperatura_max_c"] = df["temp_max_c_era5"]
df["temperatura_min_c"] = df["temp_min_c_era5"]

# Umbrales OMM
df["lluvia_medible"] = (df["precipitacion_mm"] >= 0.1).astype(int)
df["lluvia_significativa_5mm"] = (df["precipitacion_mm"] >= 5.0).astype(int)
df["lluvia_fuerte_20mm"] = (df["precipitacion_mm"] >= 20.0).astype(int)
df["es_dia_caluroso_30c"] = (df["temperatura_max_c"] >= 30.0).astype(int)
df["es_dia_frio_10c"] = (df["temperatura_min_c"] <= 10.0).astype(int)

print(f"Master Dataset creado: {df.shape[0]} filas x {df.shape[1]} columnas.")
df[["fecha", "precipitacion_mm", "temperatura_media_c", "es_dia_habil", "es_covid_aspo"]].head()

Master Dataset creado: 1461 filas x 32 columnas.


,fecha,precipitacion_mm,temperatura_media_c,es_dia_habil,es_covid_aspo
0,2018-01-01,0.0,22.1,1,0
1,2018-01-02,0.6,25.3,1,0
2,2018-01-03,0.0,25.8,1,0
3,2018-01-04,0.6,28.8,1,0
4,2018-01-05,4.4,22.8,1,0


## 6. Exportación y Resumen Estadístico de Precipitaciones

In [ ]:
out_dir = Path("../datos")
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / "clima_diario_caba_2018_2021_consolidado.csv"
df.to_csv(out_csv, index=False)

# Hash SHA-256 para trazabilidad científica
with open(out_csv, "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()

print("=" * 60)
print("DATASET GENERADO EXITOSAMENTE")
print("=" * 60)
print(f"Archivo: {out_csv}")
print(f"Observaciones: {len(df)} días (desde {df['fecha'].min().date()} al {df['fecha'].max().date()})")
print(f"Firma criptográfica SHA-256: {sha256}")
print("=" * 60)
print(f"- Días hábiles: {df['es_dia_habil'].sum()} ({df['es_dia_habil'].mean():.1%})")
print(f"- Días con precipitación medible (>0.1 mm): {df['lluvia_medible'].sum()} ({df['lluvia_medible'].mean():.1%})")
print(f"- Días con lluvia moderada (>= 5 mm): {df['lluvia_significativa_5mm'].sum()} ({df['lluvia_significativa_5mm'].mean():.1%})")
print(f"- Días con tormenta fuerte (>= 20 mm): {df['lluvia_fuerte_20mm'].sum()} ({df['lluvia_fuerte_20mm'].mean():.1%})")
print(f"- Precipitación máxima en 24h: {df['precipitacion_mm'].max():.1f} mm")
print(f"- Precipitación promedio diaria: {df['precipitacion_mm'].mean():.2f} mm/día")

DATASET GENERADO EXITOSAMENTE
Archivo: ..\datos\clima_diario_caba_2018_2021_consolidado.csv
Observaciones: 1461 días (desde 2018-01-01 al 2021-12-31)
Firma criptográfica SHA-256: 652a5e2cb4ae7c8a41197cc8a2cd89135927bf41f911580c40b0863ad75cc592
- Días hábiles: 1045 (71.5%)
- Días con precipitación medible (>0.1 mm): 594 (40.7%)
- Días con lluvia moderada (>= 5 mm): 187 (12.8%)
- Días con tormenta fuerte (>= 20 mm): 59 (4.0%)
- Precipitación máxima en 24h: 82.0 mm
- Precipitación promedio diaria: 2.78 mm/día
